In [0]:
# openai is not pre-installed on Databricks clusters.
# Run this cell first, then the kernel restarts automatically so the
# package is available in every cell that follows.
# Only needs to run once per cluster — skip on subsequent runs if the
# cluster is already running.

%pip install openai
dbutils.library.restartPython()

In [0]:
# All imports and constants live here so any reader can see the full
# dependency list in one place without hunting through the notebook.
#
# LLM connection:
#   We use the OpenAI SDK pointed at the Databricks AI Gateway.
#   The API token is read from the active Databricks session automatically —
#   no hardcoded credentials anywhere in this notebook.
#
# Catalog:
#   All tables live in globalmart.gold — the Gold layer of the medallion
#   architecture. This notebook reads only from Gold; it does not touch
#   Bronze or Silver.

import time
import json
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from openai import OpenAI

# Connect to the Databricks AI Gateway
# databricks-gpt-oss-20b is available on the Free Edition at no cost
client = OpenAI(
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
    base_url="https://4092199907892374.ai-gateway.cloud.databricks.com/mlflow/v1"
)

MODEL_NAME    = "databricks-gpt-oss-20b"
NOTEBOOK_NAME = "UC4_Executive_Business_Intelligence"
CATALOG       = "globalmart.gold"

print(f"Model     : {MODEL_NAME}")
print(f"Catalog   : {CATALOG}")
print(f"Run started: {datetime.now().isoformat()}")

In [0]:
# Three functions used by every LLM call in this notebook.
# Defined once here, called in Cells 6, 8, and 10.

def extract_text(response) -> str:
    """
    Parse the model response and return only the answer text.

    Why this is needed:
    databricks-gpt-oss-20b returns a list, not a plain string:
      [{"type": "reasoning", ...},   <- internal chain-of-thought (discard)
       {"type": "text", "text": "..."}]  <- the actual answer (keep this)

    Without this function, the raw reasoning chain would appear alongside
    the executive summary, making the output unreadable.
    """
    content = response.choices[0].message.content
    if isinstance(content, list):
        parts = [
            block["text"]
            for block in content
            if isinstance(block, dict) and block.get("type") == "text"
        ]
        return " ".join(parts).strip()
    return content.strip()


def call_llm(prompt: str, system_msg: str, retries: int = 3, wait: int = 2) -> str:
    """
    Call the LLM with automatic retry on transient failures.

    Uses exponential backoff: waits 2s, then 4s, then 6s between attempts.
    On total failure returns an error string starting with LLM_UNAVAILABLE
    so downstream validation can catch it without crashing the notebook.
    Temperature is set to 0.4 — slightly higher than classification tasks
    to allow natural narrative variation in executive prose.
    """
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user",   "content": prompt}
                ],
                max_tokens=2000,
                temperature=0.4
            )
            return extract_text(response)
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(wait * (attempt + 1))
            else:
                return f"LLM_UNAVAILABLE: {str(e)}"


def validate_output(text: str, min_words: int = 40) -> dict:
    """
    Quality-check the LLM output before writing it to the Gold layer.

    Catches three failure modes:
      too_short       — response was truncated (under 40 words)
      llm_call_failed — all retries failed (starts with LLM_UNAVAILABLE)
      refusal_detected — model refused the task instead of answering

    Returns a dict with:
      text             — the raw LLM output
      llm_check        — PASS or REVIEW
      llm_check_detail — comma-separated list of issues, or "none"

    REVIEW outputs are still written to the Gold table but flagged so
    analysts can audit them before sharing with leadership.
    """
    refusal_phrases = ["i cannot", "as an ai", "i don't have", "i'm unable"]
    issues = []

    if len(text.split()) < min_words:
        issues.append("too_short")
    if text.startswith("LLM_UNAVAILABLE"):
        issues.append("llm_call_failed")
    if any(phrase in text.lower() for phrase in refusal_phrases):
        issues.append("refusal_detected")

    return {
        "text":             text,
        "llm_check":        "PASS" if not issues else "REVIEW",
        "llm_check_detail": ", ".join(issues) if issues else "none"
    }


# System message shared across all three executive summary calls.
# It defines the LLM's role: synthesis, not data reporting.
# The model's job is to explain WHAT the numbers mean, not restate them.
SYSTEM_MSG_EXEC = (
    "You are a senior business analyst writing executive synthesis for a national retail chain. "
    "Your job is to explain what the numbers mean, not restate them. "
    "Identify the root cause behind each metric, name the specific regions or vendors involved, "
    "and end with a clear recommended action the leadership team should take. "
    "Write in plain, authoritative business English. "
    "4-6 sentences of connected prose. No bullet points, no markdown, no headers, no company name."
)

print("LLM helpers defined.")

In [0]:
# This notebook reads from three Materialized Views that the Silver→Gold
# pipeline maintains, plus two fact tables and the vendor dimension.
#
# Materialized Views (pre-aggregated, fast reads):
#   mv_revenue_by_region       — monthly revenue totals by region
#   mv_return_rate_by_vendor   — return rate per vendor
#   mv_slow_moving_products    — products with low velocity by region
#
# Fact and dimension tables (for joining vendor names):
#   fact_orders, fact_returns, dim_vendors
#
# IMPORTANT: This notebook reads Gold only. It does NOT touch Bronze or Silver.

df_revenue   = spark.table(f"{CATALOG}.mv_revenue_by_region")
df_vendor_rr = spark.table(f"{CATALOG}.mv_return_rate_by_vendor")
df_slow      = spark.table(f"{CATALOG}.mv_slow_moving_products")
df_orders    = spark.table(f"{CATALOG}.fact_orders")
df_returns   = spark.table(f"{CATALOG}.fact_returns")
df_vendors   = spark.table(f"{CATALOG}.dim_vendors")

print("Gold tables loaded:")
print(f"  mv_revenue_by_region      : {df_revenue.count():,} rows")
print(f"  mv_return_rate_by_vendor  : {df_vendor_rr.count():,} rows")
print(f"  mv_slow_moving_products   : {df_slow.count():,} rows")
print(f"  fact_orders               : {df_orders.count():,} rows")
print(f"  fact_returns              : {df_returns.count():,} rows")
print(f"  dim_vendors               : {df_vendors.count():,} rows")

In [0]:
# PURPOSE: Compute the 5 KPIs that will be passed to the LLM for the
#          revenue performance executive summary.
#
# DESIGN RULE: Never pass raw rows to the LLM. Aggregate first.
#   - A prompt with 5 KPI values produces better synthesis than 5,000 rows.
#   - Aggregating first also stays within the model's context window.
#
# KPIs computed here:
#   total_revenue    — sum of all regional sales in the latest period
#   top_region       — region with the highest revenue
#   bottom_region    — region with the lowest revenue
#   by_region        — per-region sales, order count, and MoM delta %
#
# MoM delta = (current_sales - prior_sales) / prior_sales * 100
# prior_sales = sum of all sales for the same region in earlier periods

# Step 1: Find the most recent reporting period
latest_month = df_revenue.agg(F.max("month")).collect()[0][0]

# Step 2: Current period — total sales and order count per region
revenue_current = (
    df_revenue
    .filter(F.col("month") == latest_month)
    .groupBy("region")
    .agg(
        F.sum("total_sales") .alias("total_sales"),
        F.sum("order_count") .alias("order_count")
    )
    .orderBy(F.col("total_sales").desc())
    .collect()
)

# Step 3: Prior periods — total sales per region (for MoM comparison)
prior_by_region = {
    r["region"]: r["prior_sales"]
    for r in df_revenue
    .filter(F.col("month") < latest_month)
    .groupBy("region")
    .agg(F.sum("total_sales").alias("prior_sales"))
    .collect()
}

# Step 4: Build per-region KPI dicts with MoM delta
revenue_kpis = []
for r in revenue_current:
    region      = r["region"]
    current_val = round(float(r["total_sales"] or 0), 2)
    prior_val   = round(float(prior_by_region.get(region, current_val) or current_val), 2)
    mom_delta   = round(((current_val - prior_val) / prior_val * 100) if prior_val > 0 else 0, 1)
    revenue_kpis.append({
        "region":        region,
        "total_sales":   current_val,
        "order_count":   int(r["order_count"] or 0),
        "mom_delta_pct": mom_delta
    })

total_revenue = round(sum(r["total_sales"] for r in revenue_kpis), 2)
top_region    = revenue_kpis[0]["region"]  if revenue_kpis else "N/A"
bottom_region = revenue_kpis[-1]["region"] if revenue_kpis else "N/A"

# Step 5: Serialise to JSON — this is what gets passed to the LLM
revenue_kpi_json = json.dumps({
    "period":        str(latest_month),
    "total_revenue": total_revenue,
    "top_region":    top_region,
    "bottom_region": bottom_region,
    "by_region":     revenue_kpis
})

print(f"Revenue KPIs — period: {latest_month}")
print(f"Total revenue: ${total_revenue:,.2f}")
print(f"{'Region':<12}  {'Sales':>12}  MoM")
for r in revenue_kpis:
    print(f"  {r['region']:<10}  ${r['total_sales']:>11,.2f}  {r['mom_delta_pct']:+.1f}%")

In [0]:
# PURPOSE: Turn the 5 aggregated revenue KPIs into an executive narrative.
#
# Prompt design:
#   The prompt opens by stating that a dashboard already shows these numbers.
#   The model is asked to explain what the numbers MEAN — not restate them.
#   Specifically it is asked to identify:
#     - Whether the overall trend signals growth, stability, or contraction
#     - What is DRIVING the gap between top and bottom regions
#     - The most likely ROOT CAUSE of declining regions
#     - What leadership should prioritise in the next 30 days
#
# This is the difference between a report and synthesis.

revenue_prompt = f"""Regional revenue performance for the period {latest_month} is shown below.

KPI DATA:
{revenue_kpi_json}

A dashboard already shows executives these numbers. What it does not show them is the story behind
the numbers. Write a 4-6 sentence executive synthesis that goes beyond the figures and explains:
1. What the overall revenue trend signals — is the business growing, flat, or contracting this period?
2. What is actually driving the gap between the top and bottom performing regions — is this a demand
   problem, a coverage problem, or a momentum problem? Name the specific regions and figures.
3. For regions showing month-over-month decline, what is the most likely root cause — market
   saturation, seasonal pullback, competitive pressure, or operational underperformance?
4. What the leadership team should prioritise in the next 30 days based on this data.

Do not restate the numbers mechanically. Explain what the numbers mean.
Plain business English. No bullet points. No markdown. No company name needed."""

revenue_summary = validate_output(call_llm(revenue_prompt, SYSTEM_MSG_EXEC))

print("\nREVENUE PERFORMANCE SUMMARY:")
print("=" * 70)
print(revenue_summary["text"])
print(f"\nllm_check: {revenue_summary['llm_check']}")

In [0]:
# PURPOSE: Compute vendor-level return rate KPIs for the LLM prompt.
#
# KPIs computed here:
#   total_vendors_analyzed    — how many vendors are in scope
#   vendors_above_20pct       — count of vendors exceeding the 20% threshold
#   avg_return_rate_overall   — fleet-wide average return rate
#   top_5_highest_return_rate — the five vendors with the worst rates
#
# Why 20% threshold?
#   Industry benchmark for retail: > 20% return rate signals a quality,
#   fulfilment, or category problem that requires active vendor management.
#
# Note: mv_return_rate_by_vendor may already contain a vendor_name column.
#   We drop it before joining to dim_vendors to avoid column name collision.

vendor_agg = (
    df_vendor_rr
    .drop("vendor_name")                         # avoid duplicate after join
    .join(df_vendors.select("vendor_id", "vendor_name"), on="vendor_id", how="left")
    .groupBy("vendor_id", "vendor_name")
    .agg(
        F.avg("return_rate_pct")   .alias("avg_return_rate"),
        F.sum("total_returns") .alias("total_returns"),
        F.sum("total_orders_sold")  .alias("total_orders")
    )
    .withColumn("return_rate_pct", F.round(F.col("avg_return_rate") * 100, 1))
    .orderBy(F.col("avg_return_rate").desc())
    .collect()
)

high_risk_vendors = [r for r in vendor_agg if float(r["avg_return_rate"] or 0) > 0.20]

avg_return_overall = round(
    sum(float(r["avg_return_rate"] or 0) for r in vendor_agg) / max(len(vendor_agg), 1) * 100, 1
)

vendor_kpi_json = json.dumps({
    "total_vendors_analyzed":    len(vendor_agg),
    "vendors_above_20pct":       len(high_risk_vendors),
    "avg_return_rate_overall":   avg_return_overall,
    "top_5_highest_return_rate": [
        {
            "vendor":      r["vendor_name"] or r["vendor_id"],
            "return_rate": float(r["return_rate_pct"] or 0)
        }
        for r in vendor_agg[:5]
    ]
})

print(f"Vendor return rate KPIs — {len(vendor_agg)} vendors analyzed")
print(f"Vendors above 20% threshold: {len(high_risk_vendors)}")
print(f"Fleet average return rate  : {avg_return_overall}%")
print()
print(f"{'Vendor':<30}  Return Rate")
for r in vendor_agg[:5]:
    name = str(r["vendor_name"] or r["vendor_id"])
    flag = " <-- above 20% threshold" if float(r["avg_return_rate"] or 0) > 0.20 else ""
    print(f"  {name:<28}  {r['return_rate_pct']}%{flag}")

In [0]:
# PURPOSE: Turn vendor return rate KPIs into a narrative that explains
#          what high return rates MEAN for the business.
#
# Prompt design:
#   Asks the model to diagnose each flagged vendor — is the high rate
#   caused by product quality, fulfilment defects, description mismatch,
#   or a category that structurally attracts returns?
#   Then asks for specific recommended actions per vendor.
#
# This is the synthesis a dashboard cannot provide.

vendor_prompt = f"""Vendor return rate data across all active suppliers is shown below.

KPI DATA:
{vendor_kpi_json}

A dashboard already shows these return rates as numbers. What executives need to know is what
those numbers mean for the business. Write a 4-6 sentence executive synthesis that explains:
1. Whether the overall return rate picture represents a contained issue or a systemic problem
   across the supply base.
2. For each vendor above the 20% threshold, what the return rate likely indicates — is this a
   product quality failure, a fulfilment or shipping defect, a product description mismatch,
   or a category that structurally attracts high returns? Name each vendor and its rate.
3. What the financial and operational consequence is of leaving high-return vendors unchanged —
   specifically the impact on reverse logistics cost, margin, and customer satisfaction.
4. What action the business should take in the next 30 days for each flagged vendor: renegotiate
   quality terms, place on probation with volume reduction, or move to alternative sourcing.

Do not restate the numbers mechanically. Explain what the return rates mean and what to do.
Plain business English. No bullet points. No markdown. No company name needed."""

vendor_summary = validate_output(call_llm(vendor_prompt, SYSTEM_MSG_EXEC))

print("\nVENDOR RETURN RATE SUMMARY:")
print("=" * 70)
print(vendor_summary["text"])
print(f"\nllm_check: {vendor_summary['llm_check']}")

In [0]:
# PURPOSE: Compute slow-moving inventory KPIs for the LLM prompt.
#
# KPIs computed here:
#   total_slow_moving_products    — total count of slow-moving SKUs
#   total_inventory_exposure_usd  — estimated dollar value tied up in unsold stock
#   worst_performing_region       — region with the most slow-moving products
#   by_region                     — per-region slow count and exposure
#
# Inventory exposure is estimated as SUM(total_sales) across slow-moving
# products in each region. NULL total_sales rows are treated as $0.
# This is a conservative floor estimate — actual holding cost is higher
# once carrying costs and markdown risk are factored in.

slow_agg = (
    df_slow
    .groupBy("region")
    .agg(
        F.count("*")
        .alias("slow_product_count"),
        F.sum(F.when(F.col("total_sales").isNotNull(), F.col("total_sales")).otherwise(0))
        .alias("estimated_exposure")
    )
    .orderBy(F.col("slow_product_count").desc())
    .collect()
)

total_slow     = sum(int(r["slow_product_count"] or 0) for r in slow_agg)
total_exposure = round(sum(float(r["estimated_exposure"] or 0) for r in slow_agg), 2)
worst_region   = slow_agg[0]["region"] if slow_agg else "N/A"

slow_kpi_json = json.dumps({
    "total_slow_moving_products":   total_slow,
    "total_inventory_exposure_usd": total_exposure,
    "worst_performing_region":      worst_region,
    "by_region": [
        {
            "region":       r["region"],
            "slow_count":   int(r["slow_product_count"] or 0),
            "exposure_usd": round(float(r["estimated_exposure"] or 0), 2)
        }
        for r in slow_agg
    ]
})

print("Slow-moving inventory KPIs")
print(f"Total slow-moving products  : {total_slow}")
print(f"Total inventory exposure    : ${total_exposure:,.2f}")
print(f"Worst performing region     : {worst_region}")
print()
print(f"{'Region':<12}  Products  Exposure ($)")
for r in slow_agg:
    print(f"  {r['region']:<10}  {r['slow_product_count']:>8}  ${float(r['estimated_exposure'] or 0):>12,.2f}")

In [0]:
# PURPOSE: Turn slow inventory KPIs into a narrative that explains
#          WHY the inventory is sitting unsold and what it COSTS to do nothing.
#
# Prompt design:
#   Opens with the point that a dashboard shows counts and exposure figures
#   but not why the inventory is unsold or the secondary costs of inaction.
#   Asks for root cause diagnosis and a specific resolution path.
#
# Secondary costs of inaction (beyond the headline exposure figure):
#   - Holding cost (warehouse space, insurance, labour)
#   - Markdown risk (the longer it sits, the deeper the discount needed)
#   - Opportunity cost (shelf space tied up by slow stock cannot hold faster-moving products)

slow_prompt = f"""Slow-moving inventory data by region is shown below.

KPI DATA:
{slow_kpi_json}

A dashboard already shows these inventory counts and exposure figures. What it does not show is
why this inventory is sitting unsold and what it costs the business to leave it there. Write a
4-6 sentence executive synthesis that explains:
1. What the pattern of slow-moving inventory across regions reveals — is this a localised problem
   or a sign of broader demand forecasting failure?
2. For the region with the highest concentration of slow-moving products, what is the most likely
   cause — a pricing issue, a vendor oversupply situation, a merchandising failure, or a
   demand mismatch between what was bought and what customers in that region actually want?
3. What the financial cost of inaction is — not just the dollar exposure figure, but the
   secondary costs: holding cost, markdown risk, and the opportunity cost of tied-up shelf space.
4. The most appropriate resolution path given the exposure size and region pattern — targeted
   clearance pricing, inter-region stock transfer, vendor buyback negotiation, or discontinuation.

Do not restate the numbers mechanically. Explain what the inventory pattern means and why it exists.
Plain business English. No bullet points. No markdown. No company name needed."""

slow_summary = validate_output(call_llm(slow_prompt, SYSTEM_MSG_EXEC))

print("\nSLOW-MOVING INVENTORY SUMMARY:")
print("=" * 70)
print(slow_summary["text"])
print(f"\nllm_check: {slow_summary['llm_check']}")

In [0]:
# PURPOSE: Persist all three executive summaries to the Gold layer table
#          globalmart.gold.ai_business_insights.
#
# Output table schema:
#   insight_type     — domain identifier: revenue_performance,
#                      vendor_return_rate, slow_moving_inventory
#   ai_summary       — the 4-6 sentence executive synthesis
#   kpi_json         — the aggregated KPI data passed to the LLM (for audit)
#   llm_check        — PASS or REVIEW (output quality flag)
#   llm_check_detail — comma-separated list of quality issues, or "none"
#   generated_at     — ISO timestamp of when the summary was generated
#
# Write mode is APPEND — each run adds new rows rather than overwriting.
# This preserves history so analysts can compare summaries across periods.
# mergeSchema = true handles any new columns added in future without errors.
#
# The kpi_json field is stored for auditability: an analyst can always
# see exactly what data was passed to the LLM to produce each summary.

insights = [
    {
        "insight_type":     "revenue_performance",
        "ai_summary":       revenue_summary["text"],
        "kpi_json":         revenue_kpi_json,
        "llm_check":        revenue_summary["llm_check"],
        "llm_check_detail": revenue_summary["llm_check_detail"],
        "generated_at":     datetime.now().isoformat()
    },
    {
        "insight_type":     "vendor_return_rate",
        "ai_summary":       vendor_summary["text"],
        "kpi_json":         vendor_kpi_json,
        "llm_check":        vendor_summary["llm_check"],
        "llm_check_detail": vendor_summary["llm_check_detail"],
        "generated_at":     datetime.now().isoformat()
    },
    {
        "insight_type":     "slow_moving_inventory",
        "ai_summary":       slow_summary["text"],
        "kpi_json":         slow_kpi_json,
        "llm_check":        slow_summary["llm_check"],
        "llm_check_detail": slow_summary["llm_check_detail"],
        "generated_at":     datetime.now().isoformat()
    }
]

schema = StructType([
    StructField("insight_type",     StringType(), True),
    StructField("ai_summary",       StringType(), True),
    StructField("kpi_json",         StringType(), True),
    StructField("llm_check",        StringType(), True),
    StructField("llm_check_detail", StringType(), True),
    StructField("generated_at",     StringType(), True)
])

df_insights = spark.createDataFrame(insights, schema=schema)

(
    df_insights.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(f"{CATALOG}.ai_business_insights")
)

print(f"Written to {CATALOG}.ai_business_insights")
display(spark.table(f"{CATALOG}.ai_business_insights").orderBy(F.col("generated_at").desc()).limit(3))

In [0]:
# PURPOSE: Demonstrate Databricks' ai_query() — a SQL-native function that
#          calls a Foundation Model directly inside a SQL query.
#          No Python, no API client, no notebook infrastructure required.
#
# HOW ai_query() WORKS:
#   ai_query('model-name', 'your prompt') returns a JSON string.
#   databricks-gpt-oss-20b specifically returns a JSON ARRAY:
#     [
#       {"type": "reasoning", "summary": [...]},  <- internal thinking (discard)
#       {"type": "text",      "text": "answer"}   <- the actual answer (keep)
#     ]
#
#   To extract just the answer text, chain two get_json_object() calls:
#     get_json_object(ai_query(...), '$[1]')       -- gets the text block object
#     get_json_object(<above>, '$.text')           -- gets the answer string
#
# DEMO 1: Per-vendor risk assessment
#   For each vendor in mv_return_rate_by_vendor, generate a one-sentence
#   plain-English assessment of whether the return rate is acceptable,
#   concerning, or critical — and name the most likely business reason.
#
# DEMO 2: Per-region revenue commentary
#   For each region in the latest month of mv_revenue_by_region, generate
#   a one-sentence trend statement with a likely driver.
#
# Both demos run live via spark.sql().display() so output is visible
# directly in the notebook output panel.

print("=" * 70)
print("DEMO 1 — ai_query() on vendor return rates")
print("Reads: globalmart.gold.mv_return_rate_by_vendor")
print("=" * 70)

spark.sql(f"""
SELECT
    vendor_id,
    vendor_name,
    ROUND(return_rate_pct * 100, 1)                     AS return_rate_pct,
    get_json_object(
        get_json_object(
            ai_query(
                '{MODEL_NAME}',
                CONCAT(
                    'Vendor ', vendor_name,
                    ' has a return rate of ', CAST(ROUND(return_rate_pct * 100, 1) AS STRING), '%. ',
                    'In one sentence, assess whether this is acceptable (under 15%), ',
                    'concerning (15-25%), or critical (above 25%) for a national retail chain, ',
                    'and state the most likely business reason: ',
                    'product quality, fulfilment error, or category mismatch. ',
                    'Plain English only. No markdown.'
                )
            ),
            '$[1]'
        ),
        '$.text'
    )                                               AS risk_assessment
FROM {CATALOG}.mv_return_rate_by_vendor
ORDER BY return_rate_pct DESC
LIMIT 5
""").display()

print()
print("=" * 70)
print("DEMO 2 — ai_query() on revenue performance by region")
print("Reads: globalmart.gold.mv_revenue_by_region (latest month only)")
print("=" * 70)

spark.sql(f"""
SELECT
    region,
    ROUND(total_sales, 0)                           AS total_sales,
    order_count,
    get_json_object(
        get_json_object(
            ai_query(
                '{MODEL_NAME}',
                CONCAT(
                    'The ', region, ' region generated $',
                    CAST(ROUND(total_sales, 0) AS STRING),
                    ' across ', CAST(order_count AS STRING), ' orders this period. ',
                    'In one sentence, state whether this signals growth, stability, or concern ',
                    'for a national retail chain, and identify one likely driver. ',
                    'Plain English only. No markdown.'
                )
            ),
            '$[1]'
        ),
        '$.text'
    )                                               AS revenue_commentary
FROM {CATALOG}.mv_revenue_by_region
WHERE month = (SELECT MAX(month) FROM {CATALOG}.mv_revenue_by_region)
ORDER BY total_sales DESC
""").display()

In [0]:
# PURPOSE: Append one row to the pipeline run log so operations can query:
#   - When did UC4 last run?
#   - How many domains were processed?
#   - Did any LLM outputs need manual review?
#
# A run with review_count > 0 gets status PARTIAL_REVIEW so downstream
# monitoring alerts can catch it without reading the full output table.

review_count = sum(
    1 for s in [revenue_summary, vendor_summary, slow_summary]
    if s["llm_check"] == "REVIEW"
)

run_log = [{
    "notebook":          NOTEBOOK_NAME,
    "run_timestamp":     datetime.now().isoformat(),
    "records_processed": 3,
    "llm_calls_made":    3,
    "outputs_flagged":   review_count,
    "status":            "SUCCESS" if review_count == 0 else "PARTIAL_REVIEW",
    "notes":             f"domains=revenue,vendor_return_rate,slow_inventory | period={latest_month}"
}]

(
    spark.createDataFrame(run_log)
    .write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(f"{CATALOG}.pipeline_run_log")
)

print(f"Run log written. Status: {run_log[0]['status']}")
print(f"Summaries requiring review: {review_count} / 3")